In [25]:
import os
import requests
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.utils as vutils
from PIL import Image
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cpu


## Hiperparámetros

In [26]:
Z_DIM = 100
IMG_SIZE = 64
IMG_CHANNELS = 3
FEATURES_G = 64
FEATURES_D = 64

BATCH_SIZE = 32
NUM_EPOCHS = 50
LR = 2e-4
BETAS = (0.5, 0.999)

DATA_DIR = "pokemon_sprites"
SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)

## Construcción del dataset

Si ya cuenta con el script proporcionado en la plataforma para construir el dataset,
utilícelo en su lugar (debe dejar las 898 imágenes PNG en `DATA_DIR`). A continuación se
incluye una alternativa autocontenida que descarga los sprites directamente desde el
repositorio público de GitHub de PokeAPI (`PokeAPI/sprites`), por si no cuenta con el
script o desea reproducir el dataset desde cero.

In [27]:
def download_pokemon_sprites(data_dir=DATA_DIR, n_pokemon=898):
    """
    Descarga los sprites frontales estandar desde el repositorio de GitHub de PokeAPI.
    Fuente: https://github.com/PokeAPI/sprites
    """
    os.makedirs(data_dir, exist_ok=True)
    base_url = "https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon"
    downloaded = 0
    for pid in range(1, n_pokemon + 1):
        out_path = os.path.join(data_dir, f"{pid}.png")
        if os.path.exists(out_path):
            downloaded += 1
            continue
        url = f"{base_url}/{pid}.png"
        try:
            resp = requests.get(url, timeout=10)
            if resp.status_code == 200 and len(resp.content) > 0:
                with open(out_path, "wb") as f:
                    f.write(resp.content)
                downloaded += 1
            else:
                print(f"No se pudo descargar #{pid}: HTTP {resp.status_code}")
        except Exception as e:
            print(f"Error descargando #{pid}: {e}")
        if pid % 100 == 0:
            print(f"Progreso: {pid}/{n_pokemon}")
    print(f"Descarga completa: {downloaded}/{n_pokemon} imagenes en '{data_dir}'")

# Descomente la siguiente linea para descargar el dataset (puede tardar varios minutos)
# download_pokemon_sprites()

In [ ]:
class PokemonSpriteDataset(Dataset):
    def __init__(self, data_dir=DATA_DIR, img_size=IMG_SIZE):
        self.paths = sorted([
            os.path.join(data_dir, f) for f in os.listdir(data_dir)
            if f.lower().endswith(".png")
        ])
        assert len(self.paths) > 0, (
            f"No se encontraron imagenes en '{data_dir}'. "
            "Ejecute download_pokemon_sprites()"
        )
        self.transform = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize([0.5] * 3, [0.5] * 3),
        ])

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGBA")
        bg = Image.new("RGBA", img.size, (255, 255, 255, 255))
        composited = Image.alpha_composite(bg, img).convert("RGB")
        return self.transform(composited)


dataset = PokemonSpriteDataset()
dataloader = DataLoader(
    dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, num_workers=0
)
print(f"Dataset cargado: {len(dataset)} imagenes, {len(dataloader)} batches por epoca")

Dataset cargado: 898 imagenes, 28 batches por epoca


## Task 1.1 - Generador y Discriminador

In [29]:
class Generator(nn.Module):
    def __init__(self, z_dim=Z_DIM, img_channels=IMG_CHANNELS, features_g=FEATURES_G):
        super().__init__()
        self.net = nn.Sequential(
            # z: (batch, z_dim, 1, 1) -> (batch, features_g*8, 4, 4)
            nn.ConvTranspose2d(z_dim, features_g * 8, kernel_size=4, stride=1, padding=0, bias=False),
            nn.BatchNorm2d(features_g * 8),
            nn.ReLU(True),
            # -> (batch, features_g*4, 8, 8)
            nn.ConvTranspose2d(features_g * 8, features_g * 4, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(features_g * 4),
            nn.ReLU(True),
            # -> (batch, features_g*2, 16, 16)
            nn.ConvTranspose2d(features_g * 4, features_g * 2, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(features_g * 2),
            nn.ReLU(True),
            # -> (batch, features_g, 32, 32)
            nn.ConvTranspose2d(features_g * 2, features_g, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(features_g),
            nn.ReLU(True),
            # -> (batch, img_channels, 64, 64)
            nn.ConvTranspose2d(features_g, img_channels, kernel_size=4, stride=2, padding=1, bias=False),
            nn.Tanh(),
        )

    def forward(self, z):
        return self.net(z)


class Discriminator(nn.Module):
    def __init__(self, img_channels=IMG_CHANNELS, features_d=FEATURES_D):
        super().__init__()
        self.net = nn.Sequential(
            # (batch, 3, 64, 64) -> (batch, features_d, 32, 32)
            nn.Conv2d(img_channels, features_d, kernel_size=4, stride=2, padding=1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            # -> (batch, features_d*2, 16, 16)
            nn.Conv2d(features_d, features_d * 2, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(features_d * 2),
            nn.LeakyReLU(0.2, inplace=True),
            # -> (batch, features_d*4, 8, 8)
            nn.Conv2d(features_d * 2, features_d * 4, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(features_d * 4),
            nn.LeakyReLU(0.2, inplace=True),
            # -> (batch, features_d*8, 4, 4)
            nn.Conv2d(features_d * 4, features_d * 8, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(features_d * 8),
            nn.LeakyReLU(0.2, inplace=True),
            # -> (batch, 1, 1, 1)
            nn.Conv2d(features_d * 8, 1, kernel_size=4, stride=1, padding=0, bias=False),
            nn.Sigmoid(),
        )

    def forward(self, x):
        out = self.net(x)
        return out.view(-1)

In [ ]:
def weights_init(m): #Función para inicializar los pesos de la red, recibe como parámetro el módulo actual 
    classname = m.__class__.__name__
    if classname.find("Conv") != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find("BatchNorm") != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

In [31]:
G = Generator().to(device)
D = Discriminator().to(device)
G.apply(weights_init)
D.apply(weights_init)

# Verificacion de formas exactas
z = torch.randn(4, Z_DIM, 1, 1, device=device)
assert G(z).shape == (4, 3, 64, 64), "Forma del generador incorrecta"
assert D(torch.randn(4, 3, 64, 64, device=device)).shape == (4,), "Forma del discriminador incorrecta"
print("Formas verificadas correctamente: G -> (4, 3, 64, 64), D -> (4,)")

Formas verificadas correctamente: G -> (4, 3, 64, 64), D -> (4,)


In [32]:
print(G)

Generator(
  (net): Sequential(
    (0): ConvTranspose2d(100, 512, kernel_size=(4, 4), stride=(1, 1), bias=False)
    (1): BatchNorm2d(512, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): ConvTranspose2d(512, 256, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
    (4): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (5): ReLU(inplace=True)
    (6): ConvTranspose2d(256, 128, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
    (7): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (8): ReLU(inplace=True)
    (9): ConvTranspose2d(128, 64, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
    (10): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (11): ReLU(inplace=True)
    (12): ConvTranspose2d(64, 3, kernel_size=(4, 4), stride=(2, 2), padding=

In [33]:
print(D)

Discriminator(
  (net): Sequential(
    (0): Conv2d(3, 64, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
    (1): LeakyReLU(negative_slope=0.2, inplace=True)
    (2): Conv2d(64, 128, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
    (3): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (4): LeakyReLU(negative_slope=0.2, inplace=True)
    (5): Conv2d(128, 256, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
    (6): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (7): LeakyReLU(negative_slope=0.2, inplace=True)
    (8): Conv2d(256, 512, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
    (9): BatchNorm2d(512, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (10): LeakyReLU(negative_slope=0.2, inplace=True)
    (11): Conv2d(512, 1, kernel_size=(4, 4), stride=(1, 1), bias=False)
    (12): Sigmoid()
  )
